# Uzbek ASR Benchmark — Demo Audio

Compares my fine-tuned `whisper-small` for Uzbek against open-source and commercial models on a **21.5-minute personal demo recording** of spontaneous speech where natural pauses, self-repetitions, and code-switching to English are common. The reference transcript was manually verified. _Additional text normalization is applied to reduce subjective formatting differences for unbiased model performance comparison, such as normalizing apostrophe variants, removing ellipses and filler words._ Metrics: WER, CER, and Sequence Similarity on both normalized and raw text.

![Demo Video Comparision Results](../resources/demo_video_comparison.png)

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.7 MB/s eta 0:00:00


In [ ]:
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import torch

DATASET_DIR = os.path.join("drive", "MyDrive", "Projects")

FINAL_MODEL_PATH = os.path.join(DATASET_DIR, "whisper-uzbek-final")

MODEL_NAME = "openai/whisper-small"

In [ ]:
import psutil

# Get virtual memory statistics
mem = psutil.virtual_memory()

print(f"Total System Memory (RAM): {mem.total / (1024 ** 3):.2f} GB")
print(f"Available:                 {mem.available / (1024 ** 3):.2f} GB")
print(f"Used:                      {mem.used / (1024 ** 3):.2f} GB")
print(f"Percentage:                {mem.percent}%")

# Check CUDA availability
if torch.cuda.is_available():
    device = torch.cuda.current_device()
    print(f"✓ GPU available: {torch.cuda.get_device_name(device)}")
    print(f"✓ GPU memory: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

    # Check available memory
    print(f"✓ Memory allocated: {torch.cuda.memory_allocated(device) / 1e9:.2f} GB")
    print(f"✓ Memory reserved: {torch.cuda.memory_reserved(device) / 1e9:.2f} GB")
else:
    print("⚠ No GPU found, training will be VERY slow!")
    print("  Consider using Google Colab or similar if no GPU available")

Total System Memory (RAM): 12.67 GB
Available:                 9.60 GB
Used:                      2.75 GB
Percentage:                24.2%
✓ GPU available: Tesla T4
✓ GPU memory: 15.64 GB
✓ Memory allocated: 0.00 GB
✓ Memory reserved: 0.00 GB


In [ ]:
import gc
import re
import warnings
from pathlib import Path

# Suppress the specific deprecation warning
warnings.filterwarnings("ignore", message=".*return_token_timestamps.*")


def format_srt_time(seconds):
    """Convert seconds to SRT timestamp format: HH:MM:SS,mmm"""
    if seconds is None:
        seconds = 0
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def clean_text(text):
    text = re.sub(r"\s([,\.!?;:-])", r"\1", text)  # remove space before punctuation
    text = re.sub(r"\s'", "'", text)  # fix apostrophes
    return text.strip()


def save_srt(sentences, output_path):
    """Save sentences with timestamps to an SRT file"""
    with open(output_path, "w", encoding="utf-8") as f:
        for i, sent in enumerate(sentences, start=1):
            start = format_srt_time(sent["start"])
            end = format_srt_time(sent["end"])
            text = clean_text(sent["text"])
            f.write(f"{i}\n{start} --> {end}\n{text}\n\n")
    print(f"\n✓ SRT file saved to: {output_path}")


def transcribe(model_path: str, audio_file: str, output_file: str = "output.txt"):
    """
    Test the fine-tuned model on a sample audio file and export an SRT subtitle file.

    Args:
        model_path:   Path to fine-tuned model
        audio_file:   Path to audio file to transcribe
        output_file:  Path for the output file (default: output.txt)
    """
    max_words = 7

    def group_words_by_sentences(chunks):
        sentences = []
        current_sentence = {"words": [], "start": None, "end": None, "text": ""}
        chunks = list(chunks)  # ensure indexable

        for i, chunk in enumerate(chunks):
            raw = chunk["text"]
            start, end = chunk["timestamp"]
            end = end or start

            if current_sentence["start"] is None:
                current_sentence["start"] = start

            if (not raw.startswith(" ") or raw.startswith(" '")) and current_sentence["words"]:
                current_sentence["words"][-1] += raw.strip()
            else:
                current_sentence["words"].append(raw.strip())

            current_sentence["end"] = end

            joined = clean_text(" ".join(current_sentence["words"]))
            word_count = len(joined.split())
            last_word = joined.split()[-1]

            # Lookahead: don't flush if next token is a continuation of current word
            next_raw = chunks[i + 1]["text"] if i + 1 < len(chunks) else " "
            next_is_continuation = not next_raw.startswith(" ") or next_raw.startswith(" '")

            if (last_word.endswith((".", "!", "?")) or word_count >= max_words) and not next_is_continuation:
                current_sentence["text"] = joined
                sentences.append(current_sentence.copy())
                current_sentence = {"words": [], "start": None, "end": None, "text": ""}

        if current_sentence["words"]:
            current_sentence["text"] = clean_text(" ".join(current_sentence["words"]))
            sentences.append(current_sentence)

        return sentences

    print("\n" + "=" * 50)
    print("TRANSCRIBING")
    print("=" * 50)

    import torch
    from transformers import pipeline

    pipe = pipeline(
        "automatic-speech-recognition",
        model=model_path,
        device=0 if torch.cuda.is_available() else -1,
        # dtype=torch.float16, # Uncomment if you got OutOfMemoryError
    )

    print(f"✓ Loaded model from {model_path}")
    print(f"✓ Processing: {audio_file}")

    result = pipe(
        audio_file,
        language="uz",
        task="transcribe",
        return_timestamps="word",
        chunk_length_s=30,  # process 30-second windows
        stride_length_s=5,  # 5s overlap to avoid cutting words at edges
        batch_size=1,  # process one chunk at a time
    )

    sentences = group_words_by_sentences(result["chunks"])

    p = Path(output_file)
    txt_output = p.with_suffix(".txt")
    srt_output = p.with_suffix(".srt")

    print(f"\nTranscription: {result['text']}")
    with open(txt_output, "w", encoding="utf-8") as f:
        f.write(result["text"])
    print(f"✓ Transcript saved to: {txt_output}")

    print(f"\nSentence-based timestamps:")
    for sent in sentences:
        start_str = f"{sent['start']:.2f}s" if sent["start"] is not None else "start"
        end_str = f"{sent['end']:.2f}s" if sent["end"] is not None else "end"
        print(f"[{start_str} - {end_str}]: {sent['text']}")

    save_srt(sentences, srt_output)

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [ ]:
from scripts import uzbek_text_normalizer


def additional_text_normalization(text: str):
    """
    Applies additional text normalization to reduce subjective formatting differences for unbiased model performance comparison.

    - Normalizes apostrophe variants (', ', ʼ, ʻ, etc.) to straight apostrophe (')
    - Removes ellipses
    - Removes filler words (e.g. a-a, a a, e-e)
    - Normalizes 'speech-to-text' to 'speech to text'
    """
    text = uzbek_text_normalizer.normalize_uzbek_apostrophes(text)

    text = re.sub(r"\.{2,}|…", "", text)  # ellipses

    # Filler words
    text = re.sub(r"\ba(-a)+\b[,.]?\s*", "", text)  # a-a-a.
    text = re.sub(r"\be(-e)+\b[,.]?\s*", "", text)  # e-e,
    text = re.sub(r"\baa\b[,.]?\s*", "", text, flags=re.IGNORECASE)  # Aa, aa
    re.sub(r"\ba\b[,.]?\s*", "", text, flags=re.IGNORECASE)  # a or a a,

    text = re.sub(r"\bspeech-to-text", "speech to text", text, flags=re.IGNORECASE)

    return text

In [ ]:
demo_audio_path = os.path.join(DATASET_DIR, "uzbek_asr_demo_converted.wav")

my_model_transcription_file = "my_model_transcription.txt"

ovozify_transcription_file = "ovozify_transcription.txt"
rubaistt_transcription_file = "rubaistt_transcription.txt"
kotib_transcription_file = "kotib_transcription.txt"
AIsha_open_transcription_file = "AIsha_open_transcription.txt"
Abduxoliq_transcription_file = "Abduxoliq_transcription.txt"

google_transcription_file = "google_stt_transcription.txt"
AIsha_commercial_transcription_file = "AIsha_commercial_transcription.txt"
UzbekVoiceAI_transcription_file = "UzbekVoiceAI_transcription.txt"
narakeet_transcription_file = "narakeet_transcription.txt"
Muxlisa_transcription_file = "Muxlisa_transcription.txt"

## My Model

In [ ]:
if os.path.exists(demo_audio_path):
    transcribe(
        model_path=FINAL_MODEL_PATH,
        audio_file=demo_audio_path,
        output_file=my_model_transcription_file,
    )


TRANSCRIBING


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ Loaded model from drive/MyDrive/Projects/whisper-uzbek-final
✓ Processing: drive/MyDrive/Projects/uzbek_asr_demo_converted.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
Whisper did not predict an endi


Transcription: Assalomu alaykum hammaga, mening ism Murodov Kamilov, 80 universitetining AI engineering fakulteti magistratura talabasiman. Ushbu videoda 3.5-4 oy mobaynida ustida ishlab kelayotgan loyihani natijalarini ko'rsatmoqchi edim. Ushbu loyiha aynan o'zbek ovozini, o'zbek tekstiga o'tkazib berish uchun xizmat qiladi, ya'ni ingliz tilida aytganda transkript.slaydlar ingliz tilida, chunki aynan shu slaydlardan, deyarli shu slaydlardan foydalanib, professorlarimizga loyihani ko'rsatib bergan edik. Keyin slaydlarni ingliz tiliga o'tkazib chiqish uchun ozgincha erindim, o'zbek tiliga o'tkazib chiqish uchun erindim ozgincha. Shuning uchun slaydlar ingliz tilida, lekin o'zim o'zbek tilida ko'rsatib chiqaman. Keyinchalik aynan pastda ko'rib turgan yoziguylar bu ushbu modldan foydalanib transkarib qilingan. Keyin oxirida xato qanchalik xato darajada ishlayotganini ko'rib chiqishimiz mumkin. E'tibor berishinglar oxiridim, ya'ni meni ovozim bu modulni o'rgatishda hech qachon ishlatilmag

## Open-Source Models

In [ ]:
if os.path.exists(demo_audio_path):
    transcribe("OvozifyLabs/whisper-small-uz-v1", demo_audio_path, txt_output=ovozify_transcription_file)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


TRANSCRIBING


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/371 [00:00<?, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ Loaded model from OvozifyLabs/whisper-small-uz-v1
✓ Processing: drive/MyDrive/Projects/uzbek_asr_demo_converted.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
Whisper did not predict an endi


Transcription:  assalomu alaykum hammaga, mening ismim murod aka kamilov, postav universitetining ai engineering fakulteti magistratura talabasiman. ushbu videoda uch yarim, to'rt oy mobaynida ustida ishlab kelayotgan loyihani natijalarini ko'rsatmoqchi edim. ushbu loyiha aynan o'zbek ovozini o'zbek tekstiga o'tkazib berish uchun xizmat qiladi, ya'ni idlib tilida aytganda transkrab. a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-. keyin slaydlarni ingliz tiliga o'tkazib chiqish uchun ozgincha erindim. e, o'zbek tiliga o'tkazib chiqish uchun erindim ozgincha. a-a, shuning uchun slaydlar ingliz tilida, lekin oʻzim oʻzbek tilida koʻrsatib chiqaman. keyinchalik aynan pastda koʻrib turgan yozguylar, bu ushbu moduldan foydalanib transkribe qilingan. keyin oxirida qanchalik xato darajada ishlayotganini koʻrib chiqimiz mumkin. e'tibor berishinglar og'irdim. ya'ni meni ovozim bu modulni o'rgatishda hech qachon ishlatilmagan va o'zi savol tug'ilishi mumkin. speech to text modullari

In [ ]:
_ = transcribe("islomov/rubaistt_v2_medium", demo_audio_path, rubaistt_transcription_file)


TRANSCRIBING


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ Loaded model from islomov/rubaistt_v2_medium
✓ Processing: drive/MyDrive/Projects/uzbek_asr_demo_converted.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
Whisper did not predict an endi


Transcription: assalomu alaykum hammaga. mening ismim murod aka kamilov. tossoq universitetining ai engineering fakulteti magistraturi talabasiman. ushbu videoda uch yarim to'rt oy mobaynida ustida ishlab kelayotgan loyihani natijalarini ko'rsatmoqchi edim. ushbu loyiha aynan o'zbek ovozini o'zbek tekstiga o'tkazib berish uchun xizmat qiladi. ya'ni ingliz tilida aytganda transcribe. a. slaydlar ingliz tilida, chunki aynan shu slaydlardan, deyarli shu slaydlardan foydalanib, a. professorlarimizga loyihani ko'rsatib bergan edik. keyin slaydlarni ingliz tiliga o'tkazib chiqish uchun ozgincha erindim. e, o'zbek tiliga o'tkazib chiqish uchun erindim ozgincha. a. shuning uchun slaydlar ingliz tilida, lekin o'zim o'zbek tilida ko'rsatib chiqaman. keyinchalik aynan pastda ko'rib turgan yozguninglar, bu ushbu moduldan foydalanib, transkribe qilingan. keyin oxirida xato, qanchalik xato darajada ishlayotganini ko'rib chiqishimiz mumkin. e'tibor berishinglarni xohlardim. ya'ni meni ovozim bu modu

In [ ]:
_ = transcribe("Kotib/uzbek_stt_v1", demo_audio_path, kotib_transcription_file)


TRANSCRIBING


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ Loaded model from Kotib/uzbek_stt_v1
✓ Processing: drive/MyDrive/Projects/uzbek_asr_demo_converted.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
Whisper did not predict an endi


Transcription: assalomu alaykum hammaga. mening ismim murodukt kamilov. tog'iston universitetining ai engineering fakulteti magistratura talabasiman. ushbu videoda uch yarim, to'rt oy mobaynida ustida ishlab kelayotgan loyihani natijalarini ko'rsatmoqchi edim. ushbu loyiha aynan o'zbek ovozini o'zbek tekstiga o'tkazib berish uchun xizmat qiladi. ya'ni ingliz tilida aytganda transcript. a a, slaydlar ingliz tilida, chunki aynan shu slaydlardan, deyarli shu slaydlardan foydalanib, a a, professorlarimizga loyihani ko'rsatib bergan edik. keyin slaydlarni ingliz tiliga o'tkazib chiqish uchun ozgincha erindim. e, o'zbek tiliga o'tkazib chiqish uchun erindim ozgincha. shuning uchun slaydlar ingliz tilida, lekin o'zim o'zbek tilida ko'rsatib chiqaman. qiyinchalik aynan pastda ko'rib turgan yozig'inglar, bu ushbu moduldan foydalanib transkribe qilingan. keyin oxirida xato qanchalik xato darajada ishlayotganini ko'rib chiqishimiz mumkin. e'tibor berishinglarni xohlardim. ya'ni meni ovozim bu mo

In [ ]:
_ = transcribe("aisha-org/Whisper-Uzbek", demo_audio_path, AIsha_open_transcription_file)


TRANSCRIBING


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ Loaded model from aisha-org/Whisper-Uzbek
✓ Processing: drive/MyDrive/Projects/uzbek_asr_demo_converted.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.



Transcription: Assalomu alaykum hamdagi, mening ismim Murodovi Kamilov, Dostoev universitetining Air Engineering fakulteti magistraturi talabasiman. Ushbu vizioda uch yarim-toʻrt oy mobaynida ustida ishlab kelayotgan loyihani natijalarini koʻrsatmoqchi edim. Ushbu loyiha aynan oʻzbek ovozini oʻzbek tekstiga oʻtqizib berishni xosmat qiladi, yaʼni ingliz tilida aytganda, transkrat, slaydlar ingliz tilida, chunki aynan shu slaydadan, deyarli shu slaydadan foydalanib, professorlarimizga loyihani ko‘rsatib berilgan ediki, slaydlarni ingliz tiliga o‘tqizib chiqish uchun azguncha erindim, eya o‘zbek tiliga o‘tqizib chiqish uchun erindim, azguncha. Shuni uchun, sldlar ingliz tilida, lekin oʻzim oʻzbek tilida koʻrsatib chiqaman. Heinchalik aynan pastda koʻrib turgan yozgu ila bu ushbu modeldan foydalanib, transkript qilingan, keyin oxirida qanchalik xato darajada ishlayotganini koʻrib chiqishim mumkin, e’tibor beshiylan oxirdim, ayni mana ovozim bu modelni o‘rgatishda hech qachon ishlatilmagan

In [ ]:
_ = transcribe("AbdulxoliqMirzaev/whisper-uz-medium", demo_audio_path, Abduxoliq_transcription_file)


TRANSCRIBING


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/356 [00:00<?, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ Loaded model from AbdulxoliqMirzaev/whisper-uz-medium
✓ Processing: drive/MyDrive/Projects/uzbek_asr_demo_converted.wav

Transcription: 
✓ Transcript saved to: Abduxoliq_transcription.txt

Sentence-based timestamps:

✓ SRT file saved to: Abduxoliq_transcription.srt


## Commercial Models

In [ ]:
from google.api_core.client_options import ClientOptions
# Before running set your own PROJECT_ID to .env file (e.g. GOOGLE_CLOUD_PROJECT="your id here")

from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech

# PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
PROJECT_ID = "mirodils-project-1"


def transcribe_batch_gcs_input_inline_output_v2(
        audio_uri: str,
) -> cloud_speech.BatchRecognizeResults:
    """Transcribes audio from a Google Cloud Storage URI using the Google Cloud Speech-to-Text API.
        The transcription results are returned inline in the response.
    Args:
        audio_uri (str): The Google Cloud Storage URI of the input audio file.
            Such as gs://[BUCKET]/[FILE]
    Returns:
        cloud_speech.BatchRecognizeResults: The response containing the transcription results.
    """
    # Instantiates a client
    client = SpeechClient(
        client_options=ClientOptions(
            api_endpoint=f"us-speech.googleapis.com",
        )
    )

    config = cloud_speech.RecognitionConfig(
        auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
        language_codes=["uz-UZ"],
        model="chirp_3",
    )

    file_metadata = cloud_speech.BatchRecognizeFileMetadata(uri=audio_uri)

    request = cloud_speech.BatchRecognizeRequest(
        recognizer=f"projects/{PROJECT_ID}/locations/us/recognizers/_",
        config=config,
        files=[file_metadata],
        recognition_output_config=cloud_speech.RecognitionOutputConfig(
            inline_response_config=cloud_speech.InlineOutputConfig(),
        ),
    )

    # Transcribes the audio into text
    operation = client.batch_recognize(request=request)

    print("Waiting for operation to complete...")
    response = operation.result(timeout=None)

    return response.results[audio_uri].transcript


print(f"Processing the audio")

try:
    result = transcribe_batch_gcs_input_inline_output_v2(
        audio_uri="gs://mirodils-bucket/uzbek_asr_demo_converted.wav",
    )
    print(result)
except Exception as err:
    print(f"\nCritical error during parallel processing: {type(err).__name__}: {err}")
    raise

print(f"Google Transcription Complete!")

with open(google_transcription_file, "w", encoding="utf-8") as f:
    f.write(result.results[0].alternatives[0].transcript)

print(f"✓ Google Transcript saved to: {google_transcription_file}")

Processing the audio
Waiting for operation to complete...
results {
  alternatives {
    transcript: "Assalomu alaykum hammaga, mening ismim Murodullo Komilov. Oliy ta\'lim universitetining AI engineering fakulteti magistratura talabasiman. Ushbu videoda 3.5-4 oy mobaynida ustida ishlab kelayotgan loyihani natijalarini ko\'rsatmoqchi edim. Ushbu loyiha aynan o\'zbek ovozini o\'zbek tekstiga o\'tkazib berish uchun xizmat qiladi, ya\'ni ingliz tilida aytganda transcribe. Slaydlar ingliz tilida, chunki aynan shu slaydlardan, deyarli shu slaydlardan foydalanib professorlarimizga loyihani ko\'rsatib bergan edik. Keyin slaydlarni ingliz tiliga o\'tkazib chiqish uchun ozgincha erindim, o\'zbek tiliga o\'tkazib chiqish uchun erindim ozgincha. Shuning uchun slaydlar ingliz tilida, lekin o\'zim o\'zbek tilida ko\'rsatib chiqaman. Keyinchalik aynan pastda ko\'rib turgan yozuvinglar bu ushbu modeldan foydalanib transcribe qilingan. Keyin oxirida xato, qanchalik xato darajada ishlayotganini ko\'rib

## Manual Checking

In [ ]:
# from pathlib import Path
import pandas as pd

my_model_subtitles = ""
with open(my_model_transcription_file, "r", encoding="utf-8") as f:
    my_model_subtitles = f.read()

google_subtitles = ""
with open(google_transcription_file, "r", encoding="utf-8") as f:
    google_subtitles = f.read()

pd.DataFrame(
    data=[{"path": Path(demo_audio_path).name, "reference": my_model_subtitles, "google_stt": google_subtitles}],
).to_csv("demo_video_comparison.csv", index=False)

In [ ]:
df = pd.read_csv("demo_video_comparison_checked.csv", index_col="id")
df.drop(columns=["ref_normalized", "hyp_normalized"], inplace=True)
df.rename(columns={"reference": "my_model", "sentence_checked": "reference"}, inplace=True)
print(df["reference"][0])
df.to_csv("demo_video_comparison_checked.csv", index_label="id")
df

Assalomu alaykum hammaga, mening ismim Mirodil Kamilov, Passau universitetining AI engineering fakulteti magistratura talabasiman. Ushbu videoda 3.5-4 oy mobaynida ustida ishlab kelayotgan loyihani natijalarini ko'rsatmoqchi edim. Ushbu loyiha aynan o'zbek ovozini o'zbek tekstiga o'tkazib berish uchun xizmat qiladi, ya'ni ingliz tilida aytganda transcribe. Slaydlar ingliz tilida, chunki aynan shu slaydlardan, deyarli shu slaydlardan foydalanib, professorlarimizga loyihani ko'rsatib bergan edik. Keyin slaydlarni ingliz tiliga o'tkazib chiqish uchun ozgincha erindim. E, o'zbek tiliga o'tkazib chiqish uchun erindim ozgincha. Shuning uchun slaydlar ingliz tilida, lekin o'zim o'zbek tilida ko'rsatib chiqaman. Keyinchalik aynan pastda ko'rib turgan yozuvinglar bu ushbu modeldan foydalanib transcribe qilingan. Keyin oxirida xato qanchalik xato darajada ishlayotganini ko'rib chiqishimiz mumkin. E'tibor berishinglarni xohlar edim, ya'ni meni ovozim bu modelni o'rgatishda hech qachon ishlatilmag

,path,google_stt,my_model,reference
id,,,,
0,uzbek_asr_demo_converted.wav,"Assalomu alaykum hammaga, mening ismim Murodul...","Assalomu alaykum hammaga, mening ism Murodov K...","Assalomu alaykum hammaga, mening ismim Mirodil..."


## Comparision

In [ ]:
def print_model_comparison(model_metrics: dict[str, dict]):
    """
    Print evaluation metrics for multiple models in a properly aligned table.
    """
    model_col = max(len("Model"), max(len(name) for name in model_metrics)) + 2
    metric_col = 12

    headers = [
        "WER",
        "CER",
        "Seq Sim",
        "WER (raw)",
        "CER (raw)",
        "Seq Sim (raw)",
    ]

    total_width = model_col + len(headers) * (metric_col + 1) + 13

    print("\n" + "=" * total_width)
    print("MODEL COMPARISON")
    print("=" * total_width)
    print()

    print(
        f"{'Model':<{model_col}}"
        f"{headers[0]:>{metric_col}} "
        f"{headers[1]:>{metric_col}} "
        f"{headers[2]:>{metric_col + 4}} "
        f"{headers[3]:>{metric_col + 3}} "
        f"{headers[4]:>{metric_col + 3}} "
        f"{headers[5]:>{metric_col + 3}}"
    )
    print("-" * total_width)

    for model_name, m in model_metrics.items():
        print(
            f"{model_name:<{model_col}}"
            f"{m['wer'] * 100:>{metric_col}.2f}% "
            f"{m['cer'] * 100:>{metric_col}.2f}% "
            f"{m['sequence_similarity'] * 100:>{metric_col}.2f}% "
            f"{m['wer_raw'] * 100:>{metric_col}.2f}% "
            f"{m['cer_raw'] * 100:>{metric_col}.2f}% "
            f"{m['sequence_similarity_raw'] * 100:>{metric_col}.2f}%"
        )

    print("=" * total_width + "\n")

In [ ]:
from scripts import similarity_metrics

my_model_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=df["my_model"].tolist(),
)

# Open-Source Models
transcript = ""
with open(rubaistt_transcription_file, "r", encoding="utf-8") as f:
    transcript = f.read()
transcript = additional_text_normalization(transcript)
rubaistt_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=[transcript],
)

transcript = ""
with open(kotib_transcription_file, "r", encoding="utf-8") as f:
    transcript = f.read()
transcript = additional_text_normalization(transcript)
kotib_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=[transcript],
)

transcript = ""
with open(ovozify_transcription_file, "r", encoding="utf-8") as f:
    transcript = f.read()
transcript = additional_text_normalization(transcript)
ovozify_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=[transcript],
)

transcript = ""
with open(AIsha_open_transcription_file, "r", encoding="utf-8") as f:
    transcript = f.read()
transcript = additional_text_normalization(transcript)
AIsha_open_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=[transcript],
)

print("Open-Source Models")
print_model_comparison({
    "My Model": my_model_metrics,
    "Rubaisst_v2": rubaistt_metrics,
    "Kotib": kotib_metrics,
    "OvozifyLabs": ovozify_metrics,
    "AIsha (open)": AIsha_open_metrics,
})

# Commercial Models
google_stt_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=df["google_stt"].tolist(),
)

transcript = ""
with open(AIsha_commercial_transcription_file, "r", encoding="utf-8") as f:
    transcript = f.read()
transcript = additional_text_normalization(transcript)
AIsha_commercial_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=[transcript],
)

transcript = ""
with open(UzbekVoiceAI_transcription_file, "r", encoding="utf-8") as f:
    transcript = f.read()
transcript = additional_text_normalization(transcript)
UzbekVoiceAI_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=[transcript],
)

transcript = ""
with open(Muxlisa_transcription_file, "r", encoding="utf-8") as f:
    transcript = f.read()
transcript = additional_text_normalization(transcript)
Muxlisa_metrics = similarity_metrics.calculate_batch(
    references=df["reference"].tolist(),
    hypotheses=[transcript],
)

transcript = ""
first_10_min_reference = "Assalomu alaykum hammaga, mening ismim Mirodil Kamilov, Passau universitetining AI engineering fakulteti magistratura talabasiman. Ushbu videoda 3.5-4 oy mobaynida ustida ishlab kelayotgan loyihani natijalarini ko'rsatmoqchi edim. Ushbu loyiha aynan o'zbek ovozini o'zbek tekstiga o'tkazib berish uchun xizmat qiladi, ya'ni ingliz tilida aytganda transcribe. Slaydlar ingliz tilida, chunki aynan shu slaydlardan, deyarli shu slaydlardan foydalanib, professorlarimizga loyihani ko'rsatib bergan edik. Keyin slaydlarni ingliz tiliga o'tkazib chiqish uchun ozgincha erindim. E, o'zbek tiliga o'tkazib chiqish uchun erindim ozgincha. Shuning uchun slaydlar ingliz tilida, lekin o'zim o'zbek tilida ko'rsatib chiqaman. Keyinchalik aynan pastda ko'rib turgan yozuvinglar bu ushbu modeldan foydalanib transcribe qilingan. Keyin oxirida xato qanchalik xato darajada ishlayotganini ko'rib chiqishimiz mumkin. E'tibor berishinglarni xohlar edim, ya'ni meni ovozim bu modelni o'rgatishda hech qachon ishlatilmagan. Va o'zi savol tug'ilishi mumkin: speech to text modellarini qayerda ishlatilinadi o'zi? Nimaga kerak degan savol bo'lishi mumkin. Birinchi hayolimga kelgan narsa bu accessibility, ya'ni kar yoki qulog'i yaxshi eshitmaydigan insonlarni video yoki boshqa bir audiolarni eshitayotganlarida pastida transcribe qilib, ya'ni yozuvini, nima aytilayotganini yozuvini yozib borish imkoniyati bo'ladi. Aynan ular eshitmaganligi sababli, yaxshi eshitmaganligi sababli pastdagi yozuvni o'qib turib tushunish imkoniyatiga ega bo'lishlari mumkin. Yoki har xil nima deydi, smart homelarda voice komandalar uchun ishlatilishi mumkin. Chiroqni o'chir, chiroqni yoq. Shunaqa vazifalar bajarish imkoniyati bo'ladi. Yana real-time translation, bu misol uchun Google Translateda ham ochgan bo'lsanglar, o'sha yerda aynan ovoz orqali kiritish imkoniyati bor tekstni, ya'ni ovozini yoqib turasizlar bu yoqdan ichkarida o'sha Google o'zining modelini ishlatib turib, speech to text modelini ishlatib turib, siz gapirgan ovozni tekst ko'rinishiga o'tkazadi va u tekst ko'rinishini keyin tarjima qiladi. Shunaqa yo'llarda ishlatish imkoniyati bo'ladi. Va yana xayolimga kelgani bu aynan mana shunaqa Zoom meeting yoki darslarda audio qilib yozib olasiz va keyinchalik misol uchun uyga borganingizda yoki boshqa meeting bo'lsa, o'sha yozib olgan videoingiz orqali notelar yoki umuman hamma ichidagi ma'lumotlarni tekst ko'rinishiga o'tkazish imkoniyati bo'ladi. Keyinchalik u balki muhim nuqtalarni imtihonga tayyorlanayotganda, boshqa bo'layotganda ishlatish imkoniyati bo'ladi. Yoki yana har xil media monitoring yo'llari ham bor. Qisqasi o'ylaymanki, bu kabi modellarni ishlata bilish imkoniyatlari ko'p deb o'ylayman. Ushbu loyiha ustida ishlayotganimda ko'p muammolarni kuzatdim. Bularni asosiylaridan bittasi o'sha 34 million aholi aynan o'zbek tilida birinchi til sifatida ishlatadi statistikaga qaraganda va bu til aynan AI sohasida juda sayoz resursga ega til hisoblanadi. Bu gapimga bir yaxshi misol sifatida aynan Whisper, Open AIning Whisper modeli bor-yo'g'i o'zbek tilida 18 minutgina audio data bilan audio data ishlatilib, o'rgatilgan-da va natija albatta juda yomon 116% word error rate, buni hozir ko'ramiz word error rate nimaligini. Solishtirish, shunchaki solishtirish sifatida turk tili 4300 soat audio data ishlatilgan va taxminan 16% word error rate natijasiga erishilgan. Yana solishtirish jihatidan nemis tili 13000 soat audio data ishlatilgan, aynan whisper small modelini qurish davomida. 13000 soat va misol uchun 4300 soat audio oldida 18 minut bu albatta hech narsa emas. Yana keyingi kuzatgan muammolarimdan asosiylaridan bittasi bu o'sha o'zbek tilida read speech datasetlari ko'pligi va conversational spontaneous speechlar kamligi. Hozir tushuntiraman, ya'ni read speech deganda bu kimgadir tekst beriladi. O'sha tekstni keyin o'sha odam o'qib beradi. O'qib berilgan audio yozib olingan, o'sha o'qib berilgan audioni keyin o'sha datasetga qo'shiladi-da. Lekin bundagi datasetlar yaxshi, yaxshi, lekin ko'p odam ishlab, ko'p odam qatnashishi mumkin, har xil dialektlarni ko'rsa bo'ladi, lekin muammolardan bittasi odatda yozilgandan keyin odatda adabiy tilda yoziladi, formal bo'ladi va o'qiyotganda inson misol uchun kamroq xato qiladi-da, yoki to'xtab qolishi, pauza qilishi, boshqa tillarni qo'shib yuborishi, gapini qaytarishi kabi muammolar bo'lmaydi, lekin conversation spontaneous speech misol uchun intervyulardan, podkastlardan, har xil joylardan odam haqiqatda og'zaki gapirayotganda, bu narsa yaqqol farq qiladi-da, o'sha shu kabi datasetlar kam, lekin read speech datasetlari ko'p yetarli deb hisoblayman. Va yana kuzatgan muammolarimdan bittasi bu bor-yo'g'i beshtagina ochiq modellarni ko'rdim, qaysiki yaxshi bir natija ko'rsatayotgan va ularni deyarli barchasi, o'sha beshta modellarning deyarli barchasi adashmasam uch-to'rttasi chiqarayotganda tekstni taxmin qilayotganda hamma tekstni kichkina harflarda yozilgan, lekin nima deydi, tinish belgilari bor. Misol uchun tutuq belgisi, nuqta, vergul, bu kabi tanish belgilari bor. Lekin agar tekstni kichik harflarda, faqat kichik harflarda yozilsa, uni o'qish ham qiyin-da. Ko'pincha noqulaylikni tug'dirishi mumkin. To'g'ri, lekin bu muammoni oddiygina script, Python script bilan hal qilish mumkin. Lekin afsuski shunaqa muammolarni kuzatdim-da. Mana bu yerda word error rate nimaligi. Sizda mana ikkita tekst bor, biri sizni modelingiz taxmin qilgan tekst va ikkinchisi reference tekst, ya'ni o'zi haqiqatda to'g'ri bo'lgan tekst. Ikkalasini bir-biriga solishtirayotganda so'zlarni o'zgargan so'zlari misol uchun, taxmin qilayotganda ba'zi so'zlarni o'zgartirib yozgan bo'lishi mumkin, xato, imloviy xato qilib, misol uchun, yoki ba'zi so'zlarni o'chirib tashlagan bo'lishi mumkin, yoki ba'zi so'zlarni o'zidan, model o'zidan o'zi shunaqa eshitildi deb o'ylab qo'shib yuborgan bo'lishi mumkin. Aynan o'sha so'zlarni sonini hammasini qo'shamiz. Ularni bo'lamiz, siz to'g'ri deb bilgan tekstni so'zlar soniga o'shanda world error rate kelib chiqadi. Shuning uchun ham misol uchun 116% chiqishini sababi ham balki model o'zidan-o'zi ko'p so'zlarni qo'shgan bo'lishi mumkin. Loyihaning eng asosiy maqsadlaridan, o'zi boshlaganimni sababi AI sohasida bir amaliy, bir kattaroq bir amaliy ish qilib ko'rish va bu ish davomida yangi narsalar o'rganish edi. Undan tashqari, ya'ni qo'ldan kelganicha yaxshiroq bir o'zbek o'sha automatic speech recognition yoki speech to text modelni yaratish edi va umid qildimki, ya'ni bu narsani open source qilib, ya'ni ochiq qilib ko'rsatib, kim ishlatmoqchi bo'lsa, bemalol ishlatish imkoniyatiga ega bo'lsin deb niyat qilgandim. Va undan tashqari kichikroq modelni tuzib, undan kerak bo'lsa, oddiy odam kompyuterni oddiy protsessorida ham ishlata olish qobiliyati bo'lishini niyat qilgandim. Bu loyihaga eng kerak bo'ladigan narsa, albatta, bu data edi. Ochiq bo'lgan datasetlarni qidirib ko'rsam, aynan shu oltita dataset chiqdi. Yana bitta dataset bor edi, Googlening Fleurs dataseti, lekin bu datasetda bu yoqda foydalanmadim, bu yoqda ham qo'shmadim, chunki o'zbek tili uchun juda bir kam ma'lumot bor edi u yerda. Shularni o'zi oltita datasetni o'zini hammasini qo'shadigan bo'lsak, taxminan 1750 soat data bor ekan, aynan o'zbek tili uchun. Bu qaysidir ma'noda kam bo'lishi mumkin, chunki mana hozir aytaman, to'rtta data tepadagi ko'rib turgan to'rtta datasetinglar aynan read speech bo'yicha read speech deganda shu boya aytgandek insonlarga o'sha tekst bergan, tekstni o'qib berishgan. O'sha aynan o'sha audiosidan foydalanib, dataset qurishgan-da. Aytganimdek, bu dataset read speechda insonlar ko'p ham xato qilmaydi."
with open(narakeet_transcription_file, "r", encoding="utf-8") as f:
    transcript = f.read()
transcript = additional_text_normalization(transcript)
narakeet_metrics = similarity_metrics.calculate_batch(
    references=[first_10_min_reference],
    hypotheses=[transcript],
)

print("\nCommercial Models")
print_model_comparison({
    "Google STT": google_stt_metrics,
    "AIsha (commercial)": AIsha_commercial_metrics,
    "UzbekVoiceAI": UzbekVoiceAI_metrics,
    "narakeet*": narakeet_metrics,
    "Muxlisa": Muxlisa_metrics,
})
print("*Narakeet model from narakeet.com is checked against first 10 minutes of the audio.")

Open-Source Models

MODEL COMPARISON

Model                  WER          CER          Seq Sim       WER (raw)       CER (raw)   Seq Sim (raw)
---------------------------------------------------------------------------------------------------------
My Model             14.33%         4.25%         6.80%        17.79%         4.83%        38.25%
Rubaisst_v2          15.12%         3.83%         0.88%        29.92%         7.14%         1.13%
Kotib                15.16%         4.17%         0.88%        29.65%         7.43%         1.10%
OvozifyLabs          17.71%         5.64%         2.38%        29.87%         7.58%        16.25%
AIsha (open)         41.49%        16.66%         0.78%        53.86%        19.85%         0.93%


Commercial Models

MODEL COMPARISON

Model                        WER          CER          Seq Sim       WER (raw)       CER (raw)   Seq Sim (raw)
---------------------------------------------------------------------------------------------------------------